# Forecast / Regression Error Metrics

A practical walk-through of the core error metrics used to evaluate forecasts and regression models.

Given **actual** values $y_i$ and **predicted** values $\hat{y}_i$ for $i = 1 \dots n$:

| Metric | Formula | What it tells you |
|---|---|---|
| **Residual** ($e_i$) | $e_i = y_i - \hat{y}_i$ | Per-point error (sign matters) |
| **Mean Error (ME)** | $\frac{1}{n}\sum e_i$ | Bias — are we over/under predicting? |
| **Mean Absolute Error (MAE)** | $\frac{1}{n}\sum \lvert e_i \rvert$ | Avg magnitude of error, same units as $y$ |
| **SSE** | $\sum e_i^2$ | Total squared error |
| **MSE** | $\frac{1}{n}\sum e_i^2$ | Avg squared error, penalises big misses |
| **RMSE** | $\sqrt{\text{MSE}}$ | Like MAE but in $y$ units, sensitive to outliers |
| **MAPE** | $\frac{100}{n}\sum \left\lvert \frac{e_i}{y_i} \right\rvert$ | Avg % error, scale-free |


## 0. Setup

In [2]:
import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
print("pandas", pd.__version__)

pandas 2.3.3


## 1. A small DataFrame of actuals vs predictions

Imagine these are monthly sales (actual) and our model's forecast for each month.

In [3]:
df = pd.DataFrame({
    "month":     pd.date_range("2024-01-01", periods=8, freq="MS"),
    "actual":    [100, 120, 135, 150, 162, 158, 170, 185],
    "predicted": [ 98, 125, 130, 145, 170, 150, 168, 190],
})
df

,month,actual,predicted
0,2024-01-01,100,98
1,2024-02-01,120,125
2,2024-03-01,135,130
3,2024-04-01,150,145
4,2024-05-01,162,170
5,2024-06-01,158,150
6,2024-07-01,170,168
7,2024-08-01,185,190


## 2. Residual ($e_i = y_i - \hat{y}_i$)

The building block for every other metric. Positive = we under-predicted, negative = we over-predicted.

In [4]:
df["residual"]     = df["actual"] - df["predicted"]
df["abs_error"]    = df["residual"].abs()
df["squared_err"]  = df["residual"] ** 2
df["abs_pct_err"]  = (df["residual"] / df["actual"]).abs() * 100   # in %
df

,month,actual,predicted,residual,abs_error,squared_err,abs_pct_err
0,2024-01-01,100,98,2,2,4,2.000
1,2024-02-01,120,125,-5,5,25,4.167
2,2024-03-01,135,130,5,5,25,3.704
3,2024-04-01,150,145,5,5,25,3.333
4,2024-05-01,162,170,-8,8,64,4.938
5,2024-06-01,158,150,8,8,64,5.063
6,2024-07-01,170,168,2,2,4,1.176
7,2024-08-01,185,190,-5,5,25,2.703


## 3. Compute the aggregate metrics

Each metric is just an aggregation of the per-row columns above.

In [ ]:
n = len(df)

ME   = df["residual"].mean()                 # Mean Error (bias)
MAE  = df["abs_error"].mean()                # Mean Absolute Error
SSE  = df["squared_err"].sum()               # Sum of Squared Errors
MSE  = df["squared_err"].mean()              # Mean Squared Error  (= SSE / n)
RMSE = np.sqrt(MSE)                          # Root Mean Squared Error
MAPE = df["abs_pct_err"].mean()              # Mean Absolute Percentage Error (%)

metrics = pd.Series({
    "Mean Error (ME)":                  ME,
    "Mean Absolute Error (MAE)":        MAE,
    "Sum of Squared Errors (SSE)":      SSE,
    "Mean Squared Error (MSE)":         MSE,
    "Root Mean Squared Error (RMSE)":   RMSE,
    "Mean Abs % Error (MAPE, %)":       MAPE,
})
metrics.to_frame("value")

## 4. Reusable function

Bundle it all up so you can drop in any actual/predicted arrays.

In [ ]:
def error_metrics(actual, predicted):
    actual    = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    e = actual - predicted                      # residuals
    return {
        "ME":   e.mean(),
        "MAE":  np.abs(e).mean(),
        "SSE":  np.sum(e ** 2),
        "MSE":  np.mean(e ** 2),
        "RMSE": np.sqrt(np.mean(e ** 2)),
        "MAPE": np.mean(np.abs(e / actual)) * 100,
    }

error_metrics(df["actual"], df["predicted"])

## 5. Cross-check with scikit-learn (optional)

Sanity-check the hand-rolled numbers against `sklearn.metrics`.

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_absolute_percentage_error,
)

y, yhat = df["actual"], df["predicted"]
print("MAE :", mean_absolute_error(y, yhat))
print("MSE :", mean_squared_error(y, yhat))
print("RMSE:", np.sqrt(mean_squared_error(y, yhat)))
print("MAPE:", mean_absolute_percentage_error(y, yhat) * 100, "%")

## 6. How to read them

- **ME ≈ 0** but large MAE/RMSE → errors cancel out; the model is *unbiased* but still *imprecise*.
- **RMSE ≥ MAE** always. A big gap between them means a few large errors (outliers) are dominating.
- **MAPE** is scale-free and easy to communicate ("~5% off on average"), **but** it blows up when actuals are near zero and punishes over-prediction differently from under-prediction.
- Use **SSE/MSE** when you want to penalise large misses hard (squared term); use **MAE** when every unit of error counts equally.
